# 04 - Model Tuning and Robust Comparison

Stage 4 objective: tune top Stage 3 candidates with stronger validation, compare tuned vs baseline behavior, and select one model family for final Stage 5 test evaluation.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd
from sklearn.model_selection import KFold

def _find_project_root() -> Path:
    candidates = []
    cwd = Path.cwd().resolve()
    candidates.extend([cwd, *cwd.parents])

    projects_dir = Path.home() / 'PycharmProjects'
    if projects_dir.exists():
        candidates.extend([path for path in projects_dir.iterdir() if path.is_dir()])

    for candidate in candidates:
        if (candidate / 'src' / 'config.py').is_file() and (candidate / 'data' / 'processed' / 'california_housing_clean.csv').is_file():
            return candidate

    raise RuntimeError('Could not locate project root with src/config.py and processed dataset')

PROJECT_ROOT = _find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import (
    BASELINE_RESULTS_FILE,
    PROCESSED_DATA_FILE,
    RANDOM_STATE,
    ROBUSTNESS_CV_SPLITS,
    SPLIT_METADATA_FILE,
    STAGE4_ERROR_ANALYSIS_FILE,
    TARGET_COLUMN,
    TUNED_RESULTS_FILE,
    TUNING_CV_SPLITS,
    TUNING_SUMMARY_FILE,
)
from src.data import (
    load_processed_dataset,
    save_json_report,
    save_report_dataframe,
)
from src.evaluate import (
    absolute_error_by_target_quantile,
    build_residual_frame,
    cross_validation_summary,
    evaluate_single_model_on_validation,
    regression_metrics,
    residual_summary,
)
from src.features import BaselineFeatureEngineer
from src.split import create_train_val_test_split, load_split_metadata
from src.tuning import (
    build_stage4_baseline_pipelines,
    get_stage4_candidate_model_names,
    run_stage4_tuning,
    tuning_results_frame,
    tuning_summary_payload,
)

pd.set_option('display.max_columns', 120)
pd.set_option('display.width', 160)
plt.style.use('ggplot')

## 1. Load Processed Data and Stage 3 Split Metadata

In [ ]:
df = load_processed_dataset(PROCESSED_DATA_FILE)
split_metadata = load_split_metadata(SPLIT_METADATA_FILE)

print(f'Processed data: {PROCESSED_DATA_FILE.resolve()}')
print(f'Split metadata: {SPLIT_METADATA_FILE.resolve()}')
print(f'Dataset shape: {df.shape}')
pd.Series(split_metadata, name='value')

**Conclusion:** Stage 4 explicitly reuses Stage 3 split configuration to keep experiments fully comparable.

## 2. Recreate Deterministic Train/Validation/Test Split

In [ ]:
split = create_train_val_test_split(
    df,
    target_column=TARGET_COLUMN,
    test_size=float(split_metadata['test_size']),
    val_size=float(split_metadata['validation_size']),
    random_state=int(split_metadata['random_state']),
)

print('Split sizes:')
print('X_train:', split.X_train.shape, 'y_train:', split.y_train.shape)
print('X_val  :', split.X_val.shape, 'y_val  :', split.y_val.shape)
print('X_test :', split.X_test.shape, 'y_test :', split.y_test.shape)

**Rule for Stage 4:** test split remains untouched for model selection.
All tuning and robustness checks below use training data (with CV) and validation checkpoints only.

## 3. Candidate Models from Stage 3

In [ ]:
stage3_results = pd.read_csv(BASELINE_RESULTS_FILE)
display(stage3_results)

candidate_models = get_stage4_candidate_model_names(include_ridge=True)
print('Selected for Stage 4 tuning:', candidate_models)

**Scope choice:** tune strongest tree baselines (`RandomForestRegressor`, `GradientBoostingRegressor`) and keep `Ridge` as linear reference.

## 4. Modest Feature Strategy Refinement

In [ ]:
feature_engineer_stage3 = BaselineFeatureEngineer(add_stage4_features=False)
feature_engineer_stage4 = BaselineFeatureEngineer(add_stage4_features=True)

sample_features = split.X_train.head(3)
stage3_columns = feature_engineer_stage3.transform(sample_features).columns.tolist()
stage4_columns = feature_engineer_stage4.transform(sample_features).columns.tolist()
new_columns = [column for column in stage4_columns if column not in stage3_columns]

print('Stage 3 engineered features remain unchanged.')
print('Additional Stage 4 candidate features:', new_columns)

**Feature refinement:** Stage 4 allows two extra interpretable ratios (`IncomePerPerson`, `OccupancyPerRoom`) and lets CV decide whether they help.

## 5. Stronger Validation Workflow

In [ ]:
tuning_cv = KFold(n_splits=TUNING_CV_SPLITS, shuffle=True, random_state=RANDOM_STATE)
robust_cv = KFold(n_splits=ROBUSTNESS_CV_SPLITS, shuffle=True, random_state=RANDOM_STATE)

print('Tuning CV:', tuning_cv)
print('Robustness CV:', robust_cv)

**Validation design:**
- 3-fold CV on training data for efficient hyperparameter search.
- 5-fold CV on training data for post-tuning robustness summary (mean/std).
- Validation split remains a clean checkpoint for model-to-model comparison.

## 6. Baseline Candidate Evaluation (Reference)

In [ ]:
baseline_pipelines = build_stage4_baseline_pipelines(random_state=RANDOM_STATE, include_ridge=True)

comparison_rows = []
val_predictions = {}

for family, pipeline in baseline_pipelines.items():
    label = f'{family}_baseline'
    val_metrics, fitted_model, y_val_pred = evaluate_single_model_on_validation(
        model_name=label,
        pipeline=pipeline,
        X_train=split.X_train,
        y_train=split.y_train,
        X_val=split.X_val,
        y_val=split.y_val,
    )
    cv_metrics = cross_validation_summary(
        pipeline=pipeline,
        X_train=split.X_train,
        y_train=split.y_train,
        cv=robust_cv,
    )

    comparison_rows.append({
        'Model': label,
        'ModelFamily': family,
        **{k: v for k, v in val_metrics.items() if k != 'Model'},
        **cv_metrics,
        'ConfigSummary': 'Stage 3 baseline defaults',
    })
    val_predictions[label] = y_val_pred


## 7. Hyperparameter Tuning on Training Data Only

In [ ]:
tuned_results = run_stage4_tuning(
    X_train=split.X_train,
    y_train=split.y_train,
    random_state=RANDOM_STATE,
    include_ridge=True,
    cv_splits=TUNING_CV_SPLITS,
)

tuning_frame = tuning_results_frame(tuned_results)
display(tuning_frame)

tuning_summary = tuning_summary_payload(
    results=tuned_results,
    random_state=RANDOM_STATE,
    cv_splits=TUNING_CV_SPLITS,
)
summary_path = save_json_report(tuning_summary, TUNING_SUMMARY_FILE)
print(f'Tuning summary saved to: {summary_path.resolve()}')

**Conclusion:** each candidate is tuned with compact search spaces, sklearn-native CV, and fixed random seed for reproducibility.

## 8. Compare Baseline vs Tuned Models

In [ ]:
for family, tuned in tuned_results.items():
    label = f'{family}_tuned'
    y_val_pred = tuned.best_estimator.predict(split.X_val)
    val_metrics = regression_metrics(split.y_val, y_val_pred)
    cv_metrics = cross_validation_summary(
        pipeline=tuned.best_estimator,
        X_train=split.X_train,
        y_train=split.y_train,
        cv=robust_cv,
    )

    comparison_rows.append({
        'Model': label,
        'ModelFamily': family,
        **val_metrics,
        **cv_metrics,
        'ConfigSummary': str(tuned.best_params),
    })
    val_predictions[label] = y_val_pred

comparison_df = pd.DataFrame(comparison_rows).sort_values('RMSE_val', ascending=True).reset_index(drop=True)
comparison_path = save_report_dataframe(comparison_df, TUNED_RESULTS_FILE)
print(f'Comparison saved to: {comparison_path.resolve()}')
comparison_df

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

comparison_df.set_index('Model')['RMSE_val'].sort_values().plot(kind='bar', ax=axes[0], color='steelblue')
axes[0].set_title('Validation RMSE: Baseline vs Tuned')
axes[0].set_ylabel('RMSE')

ordered = comparison_df.sort_values('CV_RMSE_mean')
axes[1].errorbar(
    x=range(len(ordered)),
    y=ordered['CV_RMSE_mean'],
    yerr=ordered['CV_RMSE_std'],
    fmt='o',
    capsize=4,
    color='darkred',
)
axes[1].set_xticks(range(len(ordered)), ordered['Model'], rotation=45, ha='right')
axes[1].set_title('Training CV RMSE Mean +/- Std')
axes[1].set_ylabel('CV RMSE')

plt.tight_layout()
plt.show()

**Robustness check:** compare both validation performance and CV variability to avoid selecting a model on a narrow or unstable gain.

## 9. Error Analysis: Baseline vs Tuned for Best Family

In [ ]:
tuned_only = comparison_df[comparison_df['Model'].str.endswith('_tuned')].copy()
best_tuned_row = tuned_only.iloc[tuned_only['RMSE_val'].argmin()]
best_family = best_tuned_row['ModelFamily']
best_tuned_label = best_tuned_row['Model']
best_baseline_label = f'{best_family}_baseline'

print('Best tuned candidate:', best_tuned_label)
print('Baseline counterpart:', best_baseline_label)

In [ ]:
residual_baseline = build_residual_frame(split.y_val, val_predictions[best_baseline_label])
residual_tuned = build_residual_frame(split.y_val, val_predictions[best_tuned_label])

residual_summary_df = pd.concat(
    [
        residual_summary(residual_baseline).rename(best_baseline_label),
        residual_summary(residual_tuned).rename(best_tuned_label),
    ],
    axis=1,
).T
residual_summary_df

In [ ]:
bin_baseline = absolute_error_by_target_quantile(residual_baseline, bins=5)
bin_baseline['Model'] = best_baseline_label

bin_tuned = absolute_error_by_target_quantile(residual_tuned, bins=5)
bin_tuned['Model'] = best_tuned_label

error_bins_df = pd.concat([bin_baseline, bin_tuned], ignore_index=True)
error_bins_path = save_report_dataframe(error_bins_df, STAGE4_ERROR_ANALYSIS_FILE)
print(f'Error by bin report saved to: {error_bins_path.resolve()}')
error_bins_df

In [ ]:
pivot_bins = error_bins_df.pivot(index='target_bin', columns='Model', values='abs_error_mean')
pivot_bins.plot(kind='bar', figsize=(10, 4))
plt.title('Absolute Error by Target Bin: Baseline vs Tuned')
plt.ylabel('Mean Absolute Error')
plt.xlabel('Target Value Bin')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].scatter(residual_tuned['actual'], residual_tuned['predicted'], alpha=0.25, s=12)
min_val = min(residual_tuned['actual'].min(), residual_tuned['predicted'].min())
max_val = max(residual_tuned['actual'].max(), residual_tuned['predicted'].max())
axes[0].plot([min_val, max_val], [min_val, max_val], color='black', linestyle='--', linewidth=1)
axes[0].set_title(f'Predicted vs Actual ({best_tuned_label})')
axes[0].set_xlabel('Actual')
axes[0].set_ylabel('Predicted')

axes[1].scatter(residual_tuned['predicted'], residual_tuned['residual'], alpha=0.25, s=12, color='darkorange')
axes[1].axhline(0, color='black', linestyle='--', linewidth=1)
axes[1].set_title(f'Residuals vs Predicted ({best_tuned_label})')
axes[1].set_xlabel('Predicted')
axes[1].set_ylabel('Residual (Actual - Predicted)')

plt.tight_layout()
plt.show()

## Stage 4 Conclusions

- Hyperparameter tuning used only training data with CV and fixed seeds.
- Validation split was reserved for consistent baseline-vs-tuned checkpoints.
- Test split remained untouched.
- The best tuned candidate from this notebook should advance to Stage 5 for final hold-out test evaluation.